# Saathi: an FAQ chatbot for ShopSaathi

Saathi answers customer questions in three steps:

1. **Find** the best matching FAQ in `data/shopsaathi_faqs.csv` (TF-IDF similarity).
2. **Escalate** to a human if no FAQ matches well (score below the threshold).
3. **Reply warmly** using a Groq-hosted model, using *only* the matched FAQ text (grounding).

Your Groq API key is never stored in this notebook. In Colab add it under the key icon (Secrets) as `GROQ_API_KEY`; elsewhere set the `GROQ_API_KEY` environment variable.

In [ ]:
!pip install -q groq pandas scikit-learn

In [ ]:
import os, textwrap
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

# ---- Settings ----
MODEL = "openai/gpt-oss-20b"   # Groq retires models over time: if you get a "decommissioned" error,
                               # list current ones with client.models.list() and change this line.
THRESHOLD = 0.12               # below this similarity score, hand over to a human
DATA_PATH = "data/shopsaathi_faqs.csv" if os.path.exists("data/shopsaathi_faqs.csv") else "shopsaathi_faqs.csv"

# ---- API key (from Colab Secrets or environment; never hard-code it) ----
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
print("AI is ON" if client else "No key found: Saathi will show plain FAQ answers.")

## Load the FAQs and build the search

In [ ]:
def build_index(path=DATA_PATH):
    faqs = pd.read_csv(path)
    search_text = faqs["question"] + " " + faqs["keywords"]
    vectorizer = TfidfVectorizer(stop_words="english")
    vectors = vectorizer.fit_transform(search_text)
    return faqs, vectorizer, vectors

faqs, vectorizer, faq_vectors = build_index()
print("Saathi knows", len(faqs), "FAQs in", faqs["category"].nunique(), "categories")
faqs.head()

## Step 1: find the best FAQ

In [ ]:
def find_best_faq(query):
    """Return (best FAQ row, similarity score) using the index built above."""
    scores = cosine_similarity(vectorizer.transform([query]), faq_vectors)[0]
    i = scores.argmax()
    return faqs.iloc[i], float(scores[i])

row, score = find_best_faq("How many days do I have to return a product?")
print(row["faq_id"], "|", row["category"], "| score", round(score, 4))

## Steps 2 and 3: escalate or reply

In [ ]:
SYSTEM_PROMPT = """Role: You are ShopSaathi's warm and friendly customer support assistant.
Task: Answer the customer's query using only the provided FAQ answer context.
Rules:
1. Reply in a warm, friendly voice: start with a short greeting, answer in your own words using only the facts in the FAQ answer, and end with a short offer to help further. Keep it to 2-3 sentences.
2. Do not use any outside information or make up details; if the answer is not in the provided FAQ context, politely say "Sorry, I'm not sure about that. Let me connect you to a support agent." """

def ask_saathi(customer_query):
    row, score = find_best_faq(customer_query)
    print("Customer Query :", customer_query)
    print("Matched FAQ    :", row["faq_id"], "| score", round(score, 4))
    if score < THRESHOLD:
        print("Saathi         : Sorry, I'm not sure about that. Let me connect you to a support agent.")
    elif client is None:
        print("Saathi         :", textwrap.fill(row["answer"], 80, subsequent_indent=" " * 17))
    else:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": f"Context (FAQ Answer): {row['answer']}\nCustomer Query: {customer_query}"}],
            temperature=0.2)
        print("Raw FAQ answer :", textwrap.fill(row["answer"], 80, subsequent_indent=" " * 17))
        print("Saathi reply   :", textwrap.fill(r.choices[0].message.content, 80, subsequent_indent=" " * 17))
    print("-" * 70)

## Try it

In [ ]:
ask_saathi("how many days do i get to return something?")
ask_saathi("Is cash on delivery available?")
ask_saathi("Do you sell live pets like puppies?")   # not in the FAQ: hands over to a human
ask_saathi("Do you sell gift cards?")               # answered by FAQ047

## Adding a new FAQ

Add a row to `data/shopsaathi_faqs.csv` with all five columns (`faq_id`, `category`, `question`, `answer`, `keywords`), then rebuild the search:

```python
faqs, vectorizer, faq_vectors = build_index()
```
Reusing wording from the question in `keywords` helps the match. Avoid common words such as "sell" in a new question, since they can pull unrelated customer questions toward it.